# GARCH en derivados: Waste Management (WM), su opción de conversión y el call spread

**Proyecto final de Derivados · Bono convertible con call spread (caso Molson Coors aplicado a WM)**
Adaptación a WM del notebook de clase *GARCH en derivados (TSLA y AAPL)*. Complemento a árboles binomiales (Hull, *Options, Futures and Other Derivatives*, 11.ª ed.).

| Sección del libro | Qué hace este notebook con WM |
|---|---|
| 23.1 Estimating volatility | Volatilidad histórica de WM y de su par Republic Services (RSG) |
| 23.2 EWMA | Volatilidad con pesos exponenciales (λ = 0,94) |
| 23.3 GARCH(1,1) | Modelo con reversión a la media de la varianza |
| 23.4 Choosing between the models | Log-verosimilitud, AIC/BIC y Ljung-Box |
| 23.5 Maximum likelihood methods | Estimación de ω, α, β y de λ por máxima verosimilitud |
| 23.6 Using GARCH(1,1) to forecast | Volatilidad promedio para **6 años** (vida del convertible) |
| 23.7 Correlations | Correlación WM-RSG con EWMA |
| 22.6 Monte Carlo simulation | Trayectorias de WM a 6 meses y a 6 años: ¿llega al precio de conversión y al del warrant? |
| 22.8 Back testing | Prueba de excepciones del VaR con la prueba de Kupiec |
| Cap. 13 (árboles) | Opción de conversión, note hedge y warrants valuados con árbol CRR y la volatilidad de GARCH |
| Proyecto | Cupón a la par del convertible (árbol Tsiveriotis-Fernandes) con la volatilidad de GARCH |

**Términos del convertible (del reporte):** USD 750 mm, 6 años, prima de conversión 25 %, warrants al 60 %, precio de WM
al 21-oct-2026 de USD 229.58 (LSEG), tasa libre de riesgo 4.957 %, YTM de WM a 6 años 5.649 %, dividendo 1.63 %.

**Cómo correrlo en Colab:** *Entorno de ejecución → Ejecutar todo*. Si Yahoo Finance no responde,
el código usa datos **simulados** (lo avisa en pantalla) para que el flujo siempre corra.
Material educativo: no es una recomendación de inversión.

In [1]:
# ---- Instalación de paquetes (solo si faltan) ----
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for _mod, _pkg in [("yfinance", "yfinance"), ("arch", "arch"), ("statsmodels", "statsmodels")]:
    try:
        __import__(_mod)
    except ImportError:
        try:
            _pip(_pkg)
        except Exception as e:
            print(f"No se pudo instalar {_pkg}:", e)

In [2]:
# ---- Librerías y configuración ----
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, brentq
from scipy.signal import lfilter
from scipy.stats import norm, chi2

# Acciones: WM (emisor) y Republic Services (par del sector, para la correlación)
TICKERS = ["WM", "RSG"]
PRINCIPAL = "WM"
YEARS = 5                       # años de historia
R_FREE = (0.049182 + 0.049954) / 2   # Treasury 6 años interpolado (LSEG US5YT=RR y US7YT=RR)
Y_WM = 0.05649                  # YTM de WM a 6 años (LSEG Issuer Curve)
DIV_YIELD = {"WM": 0.0163, "RSG": 0.0114}   # LSEG Overview
HORIZON_DAYS = 126              # pronóstico corto del precio: ~6 meses
N_SIM = 5000                    # trayectorias de Monte Carlo
LAMBDA_EWMA = 0.94              # valor clásico de RiskMetrics
MU_ANUAL = 0.08                 # rendimiento esperado anual SUPUESTO para el pronóstico del precio
SEED = 123
UMBRAL_SEÑAL = 0.10             # 10 % de diferencia entre GARCH y volatilidad implícita

# Términos del convertible (reporte WM_Proyecto_Final)
S_EMISION = 229.58              # cierre oficial de WM el 21-oct-2026 (LSEG)
T_CONV = 6.0                    # años
PRIMA_CONV, PRIMA_WARR = 0.25, 0.60
MONTO = 750.0                   # USD mm
F = 1000.0                      # valor nominal por bono
K_CONV = S_EMISION * (1 + PRIMA_CONV)
K_WARR = S_EMISION * (1 + PRIMA_WARR)
N_ACC = F / K_CONV              # acciones por bono
ACC_MM = MONTO / K_CONV         # mm de acciones subyacentes
IV_LSEG = 0.227453              # vol. implícita LSEG, calls ene-2029 strike 290
VOL_WARR_LSEG = 0.2104          # vol. histórica 90 días LSEG (la que usa el caso para el warrant)

# Valores de respaldo si no hay datos de mercado
FALLBACK_PRICE = {"WM": 229.58, "RSG": 233.40}
FALLBACK_IV = {"WM": IV_LSEG, "RSG": 0.22}
COLORES = {"WM": "#1F3864", "RSG": "darkorange"}

SAVE_FIGS = os.environ.get("SAVE_FIGS")      # carpeta opcional para guardar las figuras
if SAVE_FIGS:
    os.makedirs(SAVE_FIGS, exist_ok=True)

def mostrar(nombre):
    plt.tight_layout()
    if SAVE_FIGS:
        plt.savefig(f"{SAVE_FIGS}/{nombre}.png", dpi=110)
    plt.show()

print(f"Precio de conversión: {K_CONV:.2f} | strike del warrant: {K_WARR:.2f} | acciones por bono: {N_ACC:.4f}")

Precio de conversión: 286.98 | strike del warrant: 367.33 | acciones por bono: 3.4846


## 1. Datos: precios y rendimientos logarítmicos
Se usa $u_i = 100\ln(S_i/S_{i-1})$ (en %). Los modelos de volatilidad trabajan con estos rendimientos diarios.
Al final de la celda se guarda `datos/WM_precios.csv`, que el reporte en R Markdown usa en su sección GARCH.

In [3]:
def descargar_precios():
    try:
        import yfinance as yf
        fin = pd.Timestamp.today().normalize()
        ini = fin - pd.DateOffset(years=YEARS)
        df = yf.download(TICKERS, start=ini, end=fin, auto_adjust=True, progress=False, timeout=20)["Close"]
        df = df.dropna()
        if df.shape[0] > 300 and all(t in df.columns for t in TICKERS):
            return df[TICKERS], "Yahoo Finance (yfinance)"
    except Exception as e:
        print("No se pudo descargar de Yahoo Finance:", e)
    return None, None


def datos_sinteticos(n=1255):
    """Serie simulada con GARCH(1,1) para WM y RSG (solo como respaldo didáctico)."""
    rng = np.random.default_rng(SEED)
    rho = 0.7
    z = rng.standard_normal((n, 2))
    z[:, 1] = rho * z[:, 0] + np.sqrt(1 - rho ** 2) * z[:, 1]
    par = {"WM": (0.06, 0.06, 0.90), "RSG": (0.06, 0.06, 0.90)}   # (omega, alpha, beta) en %²
    cols = {}
    for j, t in enumerate(TICKERS):
        omega, alpha, beta = par[t]
        v = omega / (1 - alpha - beta)
        u = np.empty(n)
        for i in range(n):
            u[i] = np.sqrt(v) * z[i, j]
            v = omega + alpha * u[i] ** 2 + beta * v
        s = np.exp(np.cumsum(u / 100 + 0.0003))
        cols[t] = s / s[-1] * FALLBACK_PRICE[t]
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n)
    return pd.DataFrame(cols, index=idx)


precios, fuente = descargar_precios()
SINTETICO = precios is None
if SINTETICO:
    precios = datos_sinteticos()
    fuente = "DATOS SIMULADOS (no son precios reales de mercado)"
print("Fuente de datos:", fuente)
print(f"Observaciones: {len(precios)}   desde {precios.index[0].date()} hasta {precios.index[-1].date()}")

rend = 100 * np.log(precios / precios.shift(1)).dropna()      # rendimientos en %
print(precios.tail(3).round(2))

# Exportar los precios de WM para el reporte en R (sección 7)
if not SINTETICO:
    os.makedirs("datos", exist_ok=True)
    precios[[PRINCIPAL]].rename(columns={PRINCIPAL: "Close"}).rename_axis("Date").to_csv(
        "datos/WM_precios.csv", date_format="%Y-%m-%d")
    print("Guardado datos/WM_precios.csv (súbelo a la carpeta datos/ del repositorio)")

Fuente de datos: Yahoo Finance (yfinance)
Observaciones: 1254   desde 2021-11-03 hasta 2026-11-02
Ticker          WM     RSG
Date                      
2026-10-29  230.90  236.28
2026-10-30  233.45  244.04
2026-11-02  237.71  245.97
Guardado datos/WM_precios.csv (súbelo a la carpeta datos/ del repositorio)


## 2. Estimación de volatilidad (Hull 23.1)
Volatilidad histórica: desviación estándar de los últimos *m* rendimientos, anualizada con $\sqrt{252}$.
Su debilidad: da el mismo peso a todos los días y reacciona lento a los shocks.

In [4]:
def vol_anual_desde_var(v):
    """Varianza diaria en %² -> volatilidad anual en decimal."""
    return np.sqrt(252 * np.asarray(v)) / 100

vol_30 = rend.rolling(30).std() * np.sqrt(252) / 100
vol_hist_1a = rend.iloc[-252:].std() * np.sqrt(252) / 100
vol_hist_5a = rend.std() * np.sqrt(252) / 100

print(pd.DataFrame({"Último año": vol_hist_1a, "Toda la muestra": vol_hist_5a}).round(4))
print(f"\nReferencia LSEG para WM: vol. histórica 90 días 21.04 %, implícita (strike 290, ene-2029) {IV_LSEG:.2%}")

        Último año  Toda la muestra
Ticker                             
WM          0.2117           0.1935
RSG         0.2012           0.1866

Referencia LSEG para WM: vol. histórica 90 días 21.04 %, implícita (strike 290, ene-2029) 22.75%


## 3. EWMA (Hull 23.2)
$\sigma_n^2=\lambda\sigma_{n-1}^2+(1-\lambda)u_{n-1}^2$ — pesos que bajan exponencialmente.
Es un caso especial de GARCH con $\omega=0$, $\alpha=1-\lambda$ y $\beta=\lambda$.

In [5]:
def var_garch(params, u, v0=None):
    """Serie de varianzas condicionales de GARCH(1,1). v[t] usa u[t-1] y v[t-1]."""
    omega, alpha, beta = params
    if v0 is None:
        v0 = np.var(u)
    x = omega + alpha * u[:-1] ** 2
    y, _ = lfilter([1.0], [1.0, -beta], x, zi=[beta * v0])
    return np.r_[v0, y]


def var_ewma(u, lam=LAMBDA_EWMA):
    return var_garch((0.0, 1 - lam, lam), u, v0=np.var(u[:30]))


def loglik(v, u):
    """Log-verosimilitud gaussiana (Hull 23.5: maximizar sum[-ln v - u^2/v], aquí con constantes)."""
    return -0.5 * np.sum(np.log(2 * np.pi) + np.log(v) + u ** 2 / v)

## 4. GARCH(1,1) por máxima verosimilitud (Hull 23.3 y 23.5)
$$\sigma_n^2=\omega+\alpha u_{n-1}^2+\beta\sigma_{n-1}^2,\qquad V_L=\frac{\omega}{1-\alpha-\beta}$$
- **α**: cuánto pesa el shock de ayer. **β**: cuánta memoria tiene la varianza.
- **α + β** (persistencia) debe ser menor que 1; cerca de 1 los shocks tardan mucho en desaparecer.
- **V_L**: varianza de largo plazo hacia la que revierte el modelo. Para un convertible a 6 años es el dato clave.

In [6]:
def ajustar_garch(u):
    v = np.var(u)
    x0 = [v * 0.03, 0.08, 0.90]
    cons = ({"type": "ineq", "fun": lambda p: 0.9999 - p[1] - p[2]},)
    bnds = [(1e-6, None), (1e-4, 0.5), (0.30, 0.9998)]
    res = minimize(lambda p: -loglik(var_garch(p, u), u), x0, method="SLSQP",
                   bounds=bnds, constraints=cons, options={"maxiter": 500, "ftol": 1e-10})
    return res.x


M = {}   # resultados por acción
for t in TICKERS:
    u = rend[t].values
    p = ajustar_garch(u)
    v = var_garch(p, u)
    omega, alpha, beta = p
    VL = omega / (1 - alpha - beta)
    M[t] = dict(u=u, p=p, v=v, VL=VL, v_next=omega + alpha * u[-1] ** 2 + beta * v[-1])

tabla = pd.DataFrame({
    t: {
        "omega": M[t]["p"][0], "alpha": M[t]["p"][1], "beta": M[t]["p"][2],
        "alpha+beta": M[t]["p"][1] + M[t]["p"][2],
        "Vol largo plazo (anual)": vol_anual_desde_var(M[t]["VL"]),
        "Vol hoy (anual, GARCH)": vol_anual_desde_var(M[t]["v_next"]),
        "Vida media del shock (días)": np.log(0.5) / np.log(M[t]["p"][1] + M[t]["p"][2]),
    } for t in TICKERS
}).round(4)
print(tabla)

# Validación contra el paquete `arch` (debe dar valores muy parecidos)
try:
    from arch import arch_model
    for t in TICKERS:
        am = arch_model(M[t]["u"], mean="Zero", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
        print(f"{t}  paquete arch: omega={am.params['omega']:.4f} alpha={am.params['alpha[1]']:.4f} "
              f"beta={am.params['beta[1]']:.4f}  |  nuestro MLE: "
              f"{M[t]['p'][0]:.4f} {M[t]['p'][1]:.4f} {M[t]['p'][2]:.4f}")
except Exception as e:
    print("Validación con `arch` omitida:", e)

print("\nInterpretación:")
for t in TICKERS:
    a, b = M[t]["p"][1], M[t]["p"][2]
    print(f"- {t}: persistencia {a + b:.3f}. Un shock pierde la mitad de su efecto en "
          f"~{np.log(0.5) / np.log(a + b):.0f} días. La volatilidad de largo plazo es "
          f"{vol_anual_desde_var(M[t]['VL']):.1%} y hoy el modelo ve {vol_anual_desde_var(M[t]['v_next']):.1%}.")

                                 WM     RSG
omega                        0.4947  0.4360
alpha                        0.1353  0.2652
beta                         0.5428  0.4501
alpha+beta                   0.6781  0.7154
Vol largo plazo (anual)      0.1968  0.1965
Vol hoy (anual, GARCH)       0.2472  0.2457
Vida media del shock (días)  1.7841  2.0694
WM  paquete arch: omega=0.4880 alpha=0.1338 beta=0.5483  |  nuestro MLE: 0.4947 0.1353 0.5428
RSG  paquete arch: omega=0.4364 alpha=0.2649 beta=0.4498  |  nuestro MLE: 0.4360 0.2652 0.4501

Interpretación:
- WM: persistencia 0.678. Un shock pierde la mitad de su efecto en ~2 días. La volatilidad de largo plazo es 19.7% y hoy el modelo ve 24.7%.
- RSG: persistencia 0.715. Un shock pierde la mitad de su efecto en ~2 días. La volatilidad de largo plazo es 19.6% y hoy el modelo ve 24.6%.


In [7]:
# Gráfica: volatilidad histórica 30 días vs EWMA vs GARCH
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, t in zip(axs, TICKERS):
    idx = rend.index
    ax.plot(idx, vol_30[t], color="gray", alpha=0.6, lw=1, label="Histórica 30 días")
    ax.plot(idx, vol_anual_desde_var(var_ewma(M[t]["u"])), color="black", lw=1, ls="--", label="EWMA λ=0,94")
    ax.plot(idx, vol_anual_desde_var(M[t]["v"]), color=COLORES[t], lw=1.6, label="GARCH(1,1)")
    ax.axhline(vol_anual_desde_var(M[t]["VL"]), color=COLORES[t], ls=":", lw=1.2, label="Largo plazo (V_L)")
    if t == PRINCIPAL:
        ax.axhline(IV_LSEG, color="firebrick", ls="-.", lw=1.2, label="Implícita LSEG (conversión)")
    ax.set_title(f"{t}: volatilidad anualizada")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend(fontsize=8)
mostrar("01_volatilidades")
print("Lectura: GARCH sube rápido tras un shock y luego regresa hacia su nivel de largo plazo (línea punteada).")

Lectura: GARCH sube rápido tras un shock y luego regresa hacia su nivel de largo plazo (línea punteada).


## 5. ¿Qué modelo elegir? (Hull 23.4 y 23.5)
Se compara la **log-verosimilitud** (más alta es mejor), **AIC/BIC** (penalizan parámetros; más bajo es mejor) y la
**prueba de Ljung-Box**: si los residuos estandarizados $u^2/\sigma^2$ ya no tienen autocorrelación (p > 0,05),
el modelo capturó el *clustering* de volatilidad.

In [8]:
try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    def lb_pvalue(serie, lags=10):
        return float(acorr_ljungbox(serie, lags=[lags], return_df=True)["lb_pvalue"].iloc[0])
except Exception:
    def lb_pvalue(serie, lags=10):            # misma prueba, programada a mano
        x = np.asarray(serie) - np.mean(serie); n = len(x); d = np.sum(x ** 2)
        Q = n * (n + 2) * sum((np.sum(x[k:] * x[:-k]) / d) ** 2 / (n - k) for k in range(1, lags + 1))
        return float(1 - chi2.cdf(Q, lags))

filas = []
for t in TICKERS:
    u, n = M[t]["u"], len(M[t]["u"])
    # mejor λ por máxima verosimilitud
    grid = np.arange(0.80, 0.995, 0.005)
    lls = [loglik(var_ewma(u, l), u) for l in grid]
    lam_opt = grid[int(np.argmax(lls))]
    M[t]["lam_opt"] = lam_opt
    modelos = {
        "Volatilidad constante": (np.full(n, np.var(u)), 1),
        "EWMA λ=0,94": (var_ewma(u, 0.94), 0),
        f"EWMA λ óptimo ({lam_opt:.3f})": (var_ewma(u, lam_opt), 1),
        "GARCH(1,1)": (M[t]["v"], 3),
    }
    for nombre, (v, k) in modelos.items():
        ll = loglik(v, u)
        filas.append({
            "Acción": t, "Modelo": nombre, "Log-verosim.": ll,
            "AIC": 2 * k - 2 * ll, "BIC": k * np.log(n) - 2 * ll,
            "p Ljung-Box (u²/σ²)": lb_pvalue(u ** 2 / v),
        })
comparacion = pd.DataFrame(filas).round(3)
print(comparacion.to_string(index=False))
print("\nu² sin modelar (hay clustering si p es pequeño):",
      {t: round(lb_pvalue(M[t]["u"] ** 2), 4) for t in TICKERS})
print("Interpretación: el mejor modelo tiene menor AIC/BIC y p de Ljung-Box mayor a 0,05.")

Acción                Modelo  Log-verosim.      AIC      BIC  p Ljung-Box (u²/σ²)
    WM Volatilidad constante     -2026.056 4054.111 4059.244                0.184
    WM           EWMA λ=0,94     -2088.009 4176.018 4176.018                0.996
    WM EWMA λ óptimo (0.990)     -2037.344 4076.687 4081.820                0.875
    WM            GARCH(1,1)     -2008.764 4023.528 4038.928                0.968
   RSG Volatilidad constante     -1981.621 3965.243 3970.376                0.000
   RSG           EWMA λ=0,94     -1981.714 3963.429 3963.429                0.565
   RSG EWMA λ óptimo (0.985)     -1964.081 3930.162 3935.296                0.020
   RSG            GARCH(1,1)     -1945.913 3897.825 3913.225                0.821

u² sin modelar (hay clustering si p es pequeño): {'WM': 0.1842, 'RSG': 0.0}
Interpretación: el mejor modelo tiene menor AIC/BIC y p de Ljung-Box mayor a 0,05.


## 6. Pronóstico de volatilidad con GARCH(1,1) (Hull 23.6)
$$E[\sigma^2_{n+k}]=V_L+(\alpha+\beta)^k(\sigma^2_{n}-V_L)$$
La varianza esperada **revierte** hacia $V_L$. Para valorar una opción de $T$ días se usa la **volatilidad promedio**
de esos $T$ días. Para WM interesan dos plazos: el LEAPS listado más largo (~2.2 años, donde LSEG da la
implícita) y la vida del convertible (**6 años = 1,512 días hábiles**).

In [9]:
def pron_var(params, u, v, H):
    omega, alpha, beta = params
    VL = omega / (1 - alpha - beta)
    v_next = omega + alpha * u[-1] ** 2 + beta * v[-1]
    k = np.arange(H)
    return VL + (alpha + beta) ** k * (v_next - VL)


def vol_promedio(t, dias):
    """Volatilidad anual promedio esperada en los próximos `dias` días hábiles."""
    fv = pron_var(M[t]["p"], M[t]["u"], M[t]["v"], max(int(dias), 1))
    return float(np.sqrt(252 * fv.mean()) / 100)


def vol_promedio_hull(t, dias):
    """Fórmula continua de Hull: V_L + (1-e^{-aT})/(aT) (V(0)-V_L), a = ln(1/(alpha+beta))."""
    omega, alpha, beta = M[t]["p"]
    a = np.log(1 / (alpha + beta))
    V0 = M[t]["v_next"]
    return float(np.sqrt(252 * (M[t]["VL"] + (1 - np.exp(-a * dias)) / (a * dias) * (V0 - M[t]["VL"]))) / 100)


DIAS_CONV = int(T_CONV * 252)
plazos = [10, 30, 63, 126, 252, 554, DIAS_CONV]
est = pd.DataFrame({t: [vol_promedio(t, d) for d in plazos] for t in TICKERS},
                   index=[f"{d} días" for d in plazos[:-2]] + ["554 días (LEAPS ene-2029)", f"{DIAS_CONV} días (convertible)"]).round(4)
print("Volatilidad anual promedio esperada (estructura temporal GARCH):")
print(est)
VOL_GARCH_6A = vol_promedio(PRINCIPAL, DIAS_CONV)
print(f"\nControl con la fórmula continua de Hull (WM, {DIAS_CONV} días): {vol_promedio_hull(PRINCIPAL, DIAS_CONV):.4f}")
print(f"WM — GARCH 6 años: {VOL_GARCH_6A:.2%} vs. implícita LSEG {IV_LSEG:.2%}")

fig, ax = plt.subplots(figsize=(8, 4))
dd = np.arange(1, DIAS_CONV + 1)
for t in TICKERS:
    fv = pron_var(M[t]["p"], M[t]["u"], M[t]["v"], DIAS_CONV)
    ax.plot(dd / 252, vol_anual_desde_var(fv), color=COLORES[t], lw=2, label=f"{t}: volatilidad pronosticada")
    ax.axhline(vol_anual_desde_var(M[t]["VL"]), color=COLORES[t], ls=":", lw=1)
ax.plot(dd / 252, [vol_promedio(PRINCIPAL, d) for d in dd], color=COLORES[PRINCIPAL], ls="--", lw=1.2,
        label="WM: promedio hasta T (el que usa el árbol)")
ax.axhline(IV_LSEG, color="firebrick", ls="-.", lw=1, label="Implícita LSEG 22.75 %")
ax.set_xlabel("Años hacia adelante"); ax.set_title("Reversión de la volatilidad a su nivel de largo plazo")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}")); ax.legend(fontsize=8)
mostrar("02_estructura_temporal_vol")
print("Lectura: a 6 años casi todo el efecto de la volatilidad de hoy ya se disipó; manda la volatilidad de largo plazo.")

Volatilidad anual promedio esperada (estructura temporal GARCH):
                               WM     RSG
10 días                    0.2134  0.2144
30 días                    0.2026  0.2029
63 días                    0.1996  0.1996
126 días                   0.1982  0.1980
252 días                   0.1975  0.1973
554 días (LEAPS ene-2029)  0.1971  0.1968
1512 días (convertible)    0.1969  0.1966

Control con la fórmula continua de Hull (WM, 1512 días): 0.1969
WM — GARCH 6 años: 19.69% vs. implícita LSEG 22.75%
Lectura: a 6 años casi todo el efecto de la volatilidad de hoy ya se disipó; manda la volatilidad de largo plazo.


## 7. Correlación entre WM y RSG (Hull 23.7)
Con EWMA: $\text{cov}_n=\lambda\,\text{cov}_{n-1}+(1-\lambda)x_{n-1}y_{n-1}$ y $\rho_n=\text{cov}_n/(\sigma_x\sigma_y)$.
Una correlación alta con su par indica que la volatilidad de WM es sobre todo **de sector** (riesgo regulatorio,
volúmenes de residuos), no idiosincrática.

In [10]:
x, y = rend[TICKERS[0]].values, rend[TICKERS[1]].values
cov = np.empty(len(x)); cov[0] = np.mean(x[:30] * y[:30])
for i in range(1, len(x)):
    cov[i] = LAMBDA_EWMA * cov[i - 1] + (1 - LAMBDA_EWMA) * x[i - 1] * y[i - 1]
corr_ewma = cov / np.sqrt(var_ewma(x) * var_ewma(y))
corr_60 = rend[TICKERS[0]].rolling(60).corr(rend[TICKERS[1]])

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(rend.index, corr_60, color="gray", alpha=0.6, label="Móvil 60 días")
ax.plot(rend.index, corr_ewma, color="black", lw=1.6, label="EWMA λ=0,94")
ax.set_title(f"Correlación {TICKERS[0]}-{TICKERS[1]}"); ax.legend()
mostrar("03_correlacion")
print(f"Correlación actual (EWMA): {corr_ewma[-1]:.2f}  |  promedio de la muestra: {np.nanmean(corr_ewma):.2f}")

Correlación actual (EWMA): 0.80  |  promedio de la muestra: 0.84


## 8. Monte Carlo con GARCH: pronóstico del precio (Hull 22.6)
Se simulan miles de trayectorias: cada día $u_t=\sigma_t z_t$ y la varianza se actualiza con GARCH.
- **6 meses**, para WM y RSG (como en el notebook de clase).
- **6 años**, para WM: probabilidad de terminar arriba del precio de conversión (USD 287) y del strike del warrant (USD 367).
- Para **pronosticar la acción** se usa un rendimiento esperado supuesto (`MU_ANUAL`); para **valorar opciones**, deriva $(r-q)$ (Duan, 1995).

In [11]:
def simular_trayectorias(t, S0, H, n_sim, deriva_anual, rng, solo_final=False):
    omega, alpha, beta = M[t]["p"]
    v = np.full(n_sim, omega + alpha * M[t]["u"][-1] ** 2 + beta * M[t]["v"][-1])  # varianza del día n+1, en %²
    S = np.full(n_sim, float(S0)) if solo_final else np.empty((H + 1, n_sim))
    if not solo_final:
        S[0] = S0
    for k in range(1, H + 1):
        sig = np.sqrt(v)
        z = rng.standard_normal(n_sim)
        ret = deriva_anual / 252 - 0.5 * (sig / 100) ** 2 + (sig / 100) * z
        if solo_final:
            S = S * np.exp(ret)
        else:
            S[k] = S[k - 1] * np.exp(ret)
        v = omega + alpha * (sig * z) ** 2 + beta * v
    return S


rng = np.random.default_rng(SEED)
S0 = {t: float(precios[t].iloc[-1]) for t in TICKERS}
SIM = {t: simular_trayectorias(t, S0[t], HORIZON_DAYS, N_SIM, MU_ANUAL, rng) for t in TICKERS}

resumen_mc = {}
for t in TICKERS:
    ST = SIM[t][-1]
    q5, q25, q50, q75, q95 = np.percentile(ST, [5, 25, 50, 75, 95])
    r10 = SIM[t][10] / S0[t] - 1
    resumen_mc[t] = {"Precio hoy": S0[t], "P5": q5, "P25": q25, "Mediana": q50, "P75": q75, "P95": q95,
                     "Prob. sube": (ST > S0[t]).mean(), "VaR 99% 10 días": -np.percentile(r10, 1)}
print(pd.DataFrame(resumen_mc).round(3))

fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, t in zip(axs, TICKERS):
    dias = np.arange(HORIZON_DAYS + 1)
    p = np.percentile(SIM[t], [5, 25, 50, 75, 95], axis=1)
    ax.fill_between(dias, p[0], p[4], color=COLORES[t], alpha=0.15, label="Percentil 5-95")
    ax.fill_between(dias, p[1], p[3], color=COLORES[t], alpha=0.30, label="Percentil 25-75")
    ax.plot(dias, p[2], color=COLORES[t], lw=2, label="Mediana")
    ax.axhline(S0[t], color="gray", ls=":")
    ax.set_title(f"{t}: abanico de precios a {HORIZON_DAYS} días (GARCH)"); ax.set_xlabel("Días hábiles"); ax.legend(fontsize=8)
mostrar("04_abanico_precios")

                      WM      RSG
Precio hoy       237.710  245.970
P5               193.746  201.981
P25              222.782  231.257
Mediana          245.144  254.150
P75              269.639  279.610
P95              307.255  319.787
Prob. sube         0.586    0.594
VaR 99% 10 días    0.095    0.103


In [12]:
# WM a 6 años desde el precio de emisión: ¿cuándo se activan la conversión y los warrants?
rng = np.random.default_rng(SEED + 10)
SIM6 = simular_trayectorias(PRINCIPAL, S_EMISION, DIAS_CONV, 2000, MU_ANUAL, rng)   # 1,513 x 2,000
ST6 = SIM6[-1]
prob = {"P(S_T > precio de conversión $%.0f)" % K_CONV: (ST6 > K_CONV).mean(),
        "P(S_T > strike del warrant $%.0f)" % K_WARR: (ST6 > K_WARR).mean(),
        "Mediana de S_T": np.median(ST6), "P5 de S_T": np.percentile(ST6, 5), "P95 de S_T": np.percentile(ST6, 95)}
print(pd.Series(prob).round(3))

fig, ax = plt.subplots(figsize=(9, 4))
anios = np.arange(DIAS_CONV + 1) / 252
p = np.percentile(SIM6, [5, 25, 50, 75, 95], axis=1)
ax.fill_between(anios, p[0], p[4], color=COLORES[PRINCIPAL], alpha=0.12, label="Percentil 5-95")
ax.fill_between(anios, p[1], p[3], color=COLORES[PRINCIPAL], alpha=0.28, label="Percentil 25-75")
ax.plot(anios, p[2], color=COLORES[PRINCIPAL], lw=2, label="Mediana")
ax.axhline(K_CONV, color="firebrick", ls="--", label=f"Precio de conversión ${K_CONV:.0f}")
ax.axhline(K_WARR, color="darkgreen", ls="--", label=f"Strike del warrant ${K_WARR:.0f}")
ax.set_title("WM: abanico GARCH a 6 años (vida del convertible)"); ax.set_xlabel("Años"); ax.legend(fontsize=8)
mostrar("04b_abanico_6_anios")
print("Lectura: la probabilidad de superar el strike del warrant mide qué tan probable es que WM llegue a diluir.")

P(S_T > precio de conversión $287)      0.622
P(S_T > strike del warrant $367)        0.426
Mediana de S_T                        336.529
P5 de S_T                             149.729
P95 de S_T                            755.407
dtype: float64


Lectura: la probabilidad de superar el strike del warrant mide qué tan probable es que WM llegue a diluir.


## 9. Back testing del VaR (Hull 22.8)
Se calcula el VaR de 1 día con la volatilidad de cada modelo y se cuentan las **excepciones** (días en que la pérdida
fue mayor que el VaR). Con 99 % se esperan ~1 % de excepciones. La **prueba de Kupiec** dice si el número observado
es compatible con lo esperado (p > 0,05: el modelo pasa). GARCH se reestima cada 63 días (sin mirar el futuro).
Si GARCH pasa el back testing, confiamos en su volatilidad para valuar la opción de conversión.

In [13]:
def kupiec(x, N, p):
    if x == 0:
        lr = -2 * N * np.log(1 - p)
    else:
        ph = x / N
        lr = -2 * ((N - x) * np.log((1 - p) / (1 - ph)) + x * np.log(p / ph))
    return float(1 - chi2.cdf(lr, 1))


def backtest(t, nivel, inicio, refit=63):
    u = M[t]["u"]; n = len(u)
    z = norm.ppf(nivel)
    v_ew = var_ewma(u)
    var_pred = {"GARCH(1,1)": np.full(n, np.nan), "EWMA": v_ew, "Histórica 250d": np.full(n, np.nan)}
    for s in range(inicio, n, refit):
        fin = min(n, s + refit)
        p = ajustar_garch(u[:s])
        var_pred["GARCH(1,1)"][s:fin] = var_garch(p, u[:fin], v0=np.var(u[:s]))[s:fin]
    for s in range(inicio, n):
        var_pred["Histórica 250d"][s] = np.var(u[s - 250:s])
    filas = []
    for nombre, vp in var_pred.items():
        var_ = z * np.sqrt(vp[inicio:])
        exc = u[inicio:] < -var_
        N, x = len(exc), int(exc.sum())
        filas.append({"Acción": t, "Confianza": f"{nivel:.0%}", "Modelo": nombre, "Días": N, "Excepciones": x,
                      "Tasa observada": x / N, "Tasa esperada": 1 - nivel, "p Kupiec": kupiec(x, N, 1 - nivel)})
    return filas, var_pred["GARCH(1,1)"], z


filas_bt, serie_var = [], {}
inicio_bt = int(len(rend) * 0.6)
for t in TICKERS:
    for nivel in (0.95, 0.99):
        f, vg, z = backtest(t, nivel, inicio_bt)
        filas_bt += f
        if nivel == 0.99:
            serie_var[t] = (vg, z)
bt = pd.DataFrame(filas_bt)
print(bt.round(4).to_string(index=False))
print("\nInterpretación: p Kupiec > 0,05 y tasa observada cercana a la esperada = el modelo mide bien el riesgo.")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, t in zip(axs, TICKERS):
    u = M[t]["u"]; vg, z = serie_var[t]
    ax.plot(rend.index[inicio_bt:], u[inicio_bt:], color="gray", lw=0.8, label="Rendimiento diario (%)")
    ax.plot(rend.index[inicio_bt:], -z * np.sqrt(vg[inicio_bt:]), color=COLORES[t], lw=1.5, label="−VaR 99% (GARCH)")
    exc = u[inicio_bt:] < -z * np.sqrt(vg[inicio_bt:])
    ax.scatter(rend.index[inicio_bt:][exc], u[inicio_bt:][exc], color="black", zorder=3, s=18, label="Excepciones")
    ax.set_title(f"{t}: back testing del VaR 99% a 1 día"); ax.legend(fontsize=8)
mostrar("05_backtesting_var")

Acción Confianza         Modelo  Días  Excepciones  Tasa observada  Tasa esperada  p Kupiec
    WM       95%     GARCH(1,1)   502           21          0.0418           0.05    0.3882
    WM       95%           EWMA   502           20          0.0398           0.05    0.2796
    WM       95% Histórica 250d   502           21          0.0418           0.05    0.3882
    WM       99%     GARCH(1,1)   502           10          0.0199           0.01    0.0491
    WM       99%           EWMA   502           10          0.0199           0.01    0.0491
    WM       99% Histórica 250d   502            9          0.0179           0.01    0.1082
   RSG       95%     GARCH(1,1)   502           27          0.0538           0.05    0.7005
   RSG       95%           EWMA   502           23          0.0458           0.05    0.6629
   RSG       95% Histórica 250d   502           25          0.0498           0.05    0.9837
   RSG       99%     GARCH(1,1)   502            9          0.0179           0.0

## 10. Opción de conversión y call spread: árbol binomial (cap. 13) con la volatilidad de GARCH
El árbol CRR necesita **una** volatilidad; GARCH da la **volatilidad promedio esperada durante los 6 años**.
Valuamos con tres volatilidades (histórica, GARCH, implícita de LSEG):
- la **opción de conversión** (call americana con strike USD 287 y dividendo de 1.63 %) y su valor por bono,
- el **cupón que deja el bono a la par** (bono directo + opción, como en el caso),
- el **note hedge** (WM compra calls a USD 287) y los **warrants** (WM vende calls a USD 367).

In [14]:
def crr(S0_, K, r, sigma, T, N=500, tipo="put", americana=True, q=0.0):
    dt = T / N
    u_ = np.exp(sigma * np.sqrt(dt)); d_ = 1 / u_
    p = (np.exp((r - q) * dt) - d_) / (u_ - d_); disc = np.exp(-r * dt)
    sg = 1 if tipo == "call" else -1
    niveles = [None] * 3
    j = np.arange(N + 1)
    V = np.maximum(sg * (S0_ * u_ ** (N - j) * d_ ** j - K), 0.0)
    if N <= 2: niveles[N] = V
    for i in range(N - 1, -1, -1):
        k = np.arange(i + 1)
        cont = disc * (p * V[:i + 1] + (1 - p) * V[1:i + 2])
        V = np.maximum(cont, np.maximum(sg * (S0_ * u_ ** (i - k) * d_ ** k - K), 0.0)) if americana else cont
        if i <= 2: niveles[i] = V
    return V[0], niveles, u_, d_, dt


def griegas(S0_, K, r, sigma, T, N=500, tipo="put", americana=True, q=0.0):
    precio, niv, u_, d_, dt = crr(S0_, K, r, sigma, T, N, tipo, americana, q)
    f0, f1, f2 = niv[0][0], niv[1], niv[2]
    delta = (f1[0] - f1[1]) / (S0_ * u_ - S0_ * d_)
    gamma = ((f2[0] - f2[1]) / (S0_ * u_ ** 2 - S0_) - (f2[1] - f2[2]) / (S0_ - S0_ * d_ ** 2)) / (0.5 * (S0_ * u_ ** 2 - S0_ * d_ ** 2))
    theta = (f2[1] - f0) / (2 * dt) / 365
    vega = crr(S0_, K, r, sigma + 0.01, T, N, tipo, americana, q)[0] - precio
    rho = crr(S0_, K, r + 0.01, sigma, T, N, tipo, americana, q)[0] - precio
    return dict(precio=precio, delta=delta, gamma=gamma, theta=theta, vega=vega, rho=rho)


def valor_bono(cup, y=Y_WM, T=6, m=2):
    t_ = np.arange(1, int(T * m) + 1); cf = np.full(len(t_), F * cup / m); cf[-1] += F
    return float(np.sum(cf / (1 + y / m) ** t_))


def contrato_mercado(t, S0_):
    """LEAPS más largo de Yahoo con strike cercano al precio de conversión; si falla, usa la implícita de LSEG."""
    base = dict(iv=FALLBACK_IV[t], fuente="LSEG OPW: calls ene-2029, strike 290 (22.75 %)")
    if SINTETICO:
        return base
    try:
        import yfinance as yf
        tk = yf.Ticker(t)
        exp = max(tk.options, key=lambda e: pd.Timestamp(e))
        ch = tk.option_chain(exp)
        c = ch.calls.iloc[(ch.calls.strike - 1.25 * S0_).abs().argsort().iloc[0]]
        if float(c.impliedVolatility) > 0.05:
            return dict(iv=float(c.impliedVolatility), fuente=f"Yahoo Finance, call {exp} strike {c.strike:.0f}")
    except Exception:
        pass
    return base


mercado = contrato_mercado(PRINCIPAL, S0[PRINCIPAL])
VOLS = {"Histórica 1 año": float(vol_hist_1a[PRINCIPAL]), "GARCH promedio 6 años": VOL_GARCH_6A,
        "Implícita LSEG": IV_LSEG}
if mercado["iv"] != IV_LSEG:
    VOLS["Implícita Yahoo (LEAPS)"] = mercado["iv"]
q_wm = DIV_YIELD[PRINCIPAL]

filas, RES = [], {}
for etiqueta, sg in VOLS.items():
    g = griegas(S_EMISION, K_CONV, R_FREE, sg, T_CONV, 500, "call", True, q_wm)
    opcion_bono = g["precio"] * N_ACC
    cupon = brentq(lambda c: valor_bono(c) - (F - opcion_bono), -0.05, 0.10)
    hedge = g["precio"] * ACC_MM
    warr = crr(S_EMISION, K_WARR, R_FREE, sg, T_CONV, 500, "call", True, q_wm)[0] * ACC_MM
    RES[etiqueta] = dict(sigma=sg, call=g["precio"], cupon=cupon, hedge=hedge, warr=warr)
    filas.append({"Volatilidad usada": etiqueta, "σ": sg, "Call por acción": g["precio"], "Δ": g["delta"],
                  "Gamma": g["gamma"], "Vega (1 pto)": g["vega"], "Opción por bono": opcion_bono,
                  "Cupón a la par": cupon, "Note hedge (USD mm)": hedge, "Warrants (USD mm)": warr,
                  "Call spread neto (USD mm)": hedge - warr})
tabla_op = pd.DataFrame(filas).round(4)
print(tabla_op.to_string(index=False))
# Convención del caso y del reporte: note hedge con la implícita, warrant con la histórica de 90 días (LSEG)
hedge_rep = RES["Implícita LSEG"]["hedge"]
warr_rep = crr(S_EMISION, K_WARR, R_FREE, VOL_WARR_LSEG, T_CONV, 500, "call", True, q_wm)[0] * ACC_MM
NETO_REPORTE = hedge_rep - warr_rep
print(f"\nConvención del reporte (hedge a {IV_LSEG:.2%}, warrant a {VOL_WARR_LSEG:.2%}): "
      f"note hedge USD {hedge_rep:.1f} mm − warrants USD {warr_rep:.1f} mm = USD {NETO_REPORTE:.1f} mm "
      f"({NETO_REPORTE / MONTO:.1%} del principal)")
print(f"\nStrike conversión {K_CONV:.2f}, warrant {K_WARR:.2f}, plazo {T_CONV:.0f} años, S = {S_EMISION} (emisión). "
      f"Implícita de mercado: {mercado['fuente']}")
print("Interpretación: el árbol es el mismo de clase; GARCH solo cambia la volatilidad que le damos.")

      Volatilidad usada      σ  Call por acción      Δ  Gamma  Vega (1 pto)  Opción por bono  Cupón a la par  Note hedge (USD mm)  Warrants (USD mm)  Call spread neto (USD mm)
        Histórica 1 año 0.2117          40.6735 0.5305 0.0030        2.0069         141.7319          0.0283             106.2990            60.6824                    45.6165
  GARCH promedio 6 años 0.1969          37.7166 0.5227 0.0032        1.9866         131.4283          0.0304              98.5712            53.1283                    45.4429
         Implícita LSEG 0.2275          43.8252 0.5387 0.0028        1.9762         152.7142          0.0261             114.5357            68.9003                    45.6353
Implícita Yahoo (LEAPS) 0.2675          51.7149 0.5589 0.0023        1.9772         180.2068          0.0207             135.1551            89.8755                    45.2796

Convención del reporte (hedge a 22.75%, warrant a 21.04%): note hedge USD 114.5 mm − warrants USD 60.0 mm = USD 54.5 mm

## 10b. Opción americana con GARCH y árbol: ¿conviene convertir antes?
En el árbol, en cada nodo se compara **convertir ya** contra **esperar** (Hull cap. 13).
- **Prima de ejercicio anticipado** = precio americano − precio europeo.
- A diferencia de la call de clase sobre una acción sin dividendos, **WM paga 1.63 %**: si la acción sube mucho,
  al tenedor le conviene convertir antes para cobrar los dividendos. El **precio crítico** es el precio de WM por
  arriba del cual convertir hoy es óptimo.

In [15]:
sg = VOL_GARCH_6A
ca = crr(S_EMISION, K_CONV, R_FREE, sg, T_CONV, 500, "call", True, q_wm)[0]
ce = crr(S_EMISION, K_CONV, R_FREE, sg, T_CONV, 500, "call", False, q_wm)[0]
grid_s = np.linspace(K_CONV, 4 * K_CONV, 80)
ejercer = [crr(s, K_CONV, R_FREE, sg, T_CONV, 300, "call", True, q_wm)[0] - (s - K_CONV) < 1e-6 for s in grid_s]
critico = float(grid_s[np.where(ejercer)[0].min()]) if any(ejercer) else None
print(pd.DataFrame([{"σ GARCH": sg, "Call americana": ca, "Call europea": ce, "Prima ejercicio anticipado": ca - ce,
                     "% del precio": 100 * (ca - ce) / ce}]).round(4).to_string(index=False))
if critico:
    print(f"\nSi WM sube por arriba de ~USD {critico:.0f} (hoy {S_EMISION}), convertir hoy es óptimo: "
          f"los dividendos que se pierden por esperar valen más que el valor tiempo de la opción.")
else:
    print("\nEn este rango de precios nunca conviene convertir hoy: esperar vale más.")
print("Por eso el reporte valúa el convertible completo en el árbol (Tsiveriotis-Fernandes) y no con la fórmula europea.")

 σ GARCH  Call americana  Call europea  Prima ejercicio anticipado  % del precio
  0.1969         37.7166       37.7101                      0.0065        0.0173

Si WM sube por arriba de ~USD 1104 (hoy 229.58), convertir hoy es óptimo: los dividendos que se pierden por esperar valen más que el valor tiempo de la opción.
Por eso el reporte valúa el convertible completo en el árbol (Tsiveriotis-Fernandes) y no con la fórmula europea.


## 10c. Convertible completo en el árbol (Tsiveriotis-Fernandes) con la volatilidad de GARCH
El componente en efectivo (cupones y principal) se descuenta a la YTM de WM y el componente en acciones a la
libre de riesgo; en cada nodo el tenedor convierte si $n\cdot S$ supera el valor de continuar. Es el mismo modelo
del reporte en R; aquí se resuelve el cupón a la par con cada volatilidad.

In [16]:
def tf_convertible(cup, s, S=S_EMISION, prima=PRIMA_CONV, N=360, qd=DIV_YIELD["WM"]):
    nn = F / (S * (1 + prima)); dt = T_CONV / N
    up = np.exp(s * np.sqrt(dt)); dn = 1 / up; p = (np.exp((R_FREE - qd) * dt) - dn) / (up - dn)
    rc = 2 * np.log(1 + Y_WM / 2); dr = np.exp(-R_FREE * dt); dk = np.exp(-rc * dt)
    paso = int(N / (2 * T_CONV)); cpn = F * cup / 2
    j = np.arange(N + 1); conv = nn * S * up ** (N - j) * dn ** j
    V = np.maximum(conv, F + cpn); B = np.where(conv >= F + cpn, 0, F + cpn)
    for i in range(N - 1, -1, -1):
        Bc = dk * (p * B[:-1] + (1 - p) * B[1:])
        Ec = dr * (p * (V[:-1] - B[:-1]) + (1 - p) * (V[1:] - B[1:]))
        if i > 0 and i % paso == 0:
            Bc = Bc + cpn
        Vc = Bc + Ec; j = np.arange(i + 1); cv = nn * S * up ** (i - j) * dn ** j
        convierte = cv > Vc
        V = np.where(convierte, cv, Vc); B = np.where(convierte, 0, Bc)
    return V[0]


cupon_tf = {k: brentq(lambda c: tf_convertible(c, v["sigma"]) - F, -0.05, 0.10) for k, v in RES.items()}
res_tf = pd.DataFrame({"σ": {k: v["sigma"] for k, v in RES.items()},
                       "Cupón (bono + call, caso)": {k: v["cupon"] for k, v in RES.items()},
                       "Cupón (árbol T-F)": cupon_tf,
                       "Ahorro vs. bono directo (pb)": {k: (Y_WM - c) * 1e4 for k, c in cupon_tf.items()}}).round(4)
print(res_tf)
print(f"\nCaso base del reporte (implícita LSEG): cupón T-F = {cupon_tf['Implícita LSEG']:.3%}")

                              σ  Cupón (bono + call, caso)  Cupón (árbol T-F)  \
Histórica 1 año          0.2117                     0.0283             0.0264   
GARCH promedio 6 años    0.1969                     0.0304             0.0285   
Implícita LSEG           0.2275                     0.0261             0.0241   
Implícita Yahoo (LEAPS)  0.2675                     0.0207             0.0182   

                         Ahorro vs. bono directo (pb)  
Histórica 1 año                              300.9262  
GARCH promedio 6 años                        279.9240  
Implícita LSEG                               324.0970  
Implícita Yahoo (LEAPS)                      383.2380  

Caso base del reporte (implícita LSEG): cupón T-F = 2.408%


## 11. Comprobación: Monte Carlo GARCH (Duan) vs árbol (opciones europeas)
Si el árbol con la volatilidad promedio de GARCH se parece al Monte Carlo con GARCH completo (volatilidad que cambia
día a día durante 6 años), la aproximación de Hull (23.6) es razonable para la opción de conversión y el warrant.

In [17]:
rng = np.random.default_rng(SEED + 1)
S_rn = simular_trayectorias(PRINCIPAL, S_EMISION, DIAS_CONV, N_SIM * 2, R_FREE - q_wm, rng, solo_final=True)
disc = np.exp(-R_FREE * T_CONV)
comp = []
for nombre, K in (("Opción de conversión / note hedge", K_CONV), ("Warrant", K_WARR)):
    pagos = np.maximum(S_rn - K, 0)
    mc = disc * pagos.mean(); ee = disc * pagos.std() / np.sqrt(len(pagos))
    ar = crr(S_EMISION, K, R_FREE, VOL_GARCH_6A, T_CONV, 500, "call", False, q_wm)[0]
    comp.append({"Opción": nombre, "Strike": K, "MC-GARCH": mc, "Error estándar MC": ee,
                 "Árbol (σ GARCH)": ar, "Dif. %": 100 * (ar / mc - 1)})
print(pd.DataFrame(comp).round(3).to_string(index=False))
print("Interpretación: si la diferencia es de pocos puntos porcentuales, usar la volatilidad promedio de GARCH en el árbol "
      "es una buena aproximación.")

                           Opción  Strike  MC-GARCH  Error estándar MC  Árbol (σ GARCH)  Dif. %
Opción de conversión / note hedge 286.975    37.074              0.758           37.710   1.715
                          Warrant 367.328    19.811              0.586           20.328   2.606
Interpretación: si la diferencia es de pocos puntos porcentuales, usar la volatilidad promedio de GARCH en el árbol es una buena aproximación.


## 12. Estrategia: señal de volatilidad para WM como emisor
En el notebook de clase la señal decide si comprar o vender un straddle. Para WM la lógica se invierte de papel:
al emitir el convertible **WM vende volatilidad** (la opción de conversión) y con el call spread **compra** volatilidad
entre USD 287 y USD 367.
1. **Señal:** razón σ GARCH / σ implícita.
   - GARCH < implícita → el mercado paga cara la opción → **buen momento para emitir** (cupón más bajo del que justifica GARCH).
   - GARCH > implícita → la opción se vende barata → conviene subir la prima de conversión o esperar.
2. **Resultado esperado:** con trayectorias GARCH del mundo real a 6 años se estima la dilución que WM entrega al
   vencimiento, con y sin call spread.

In [18]:
ratio = VOL_GARCH_6A / IV_LSEG
if ratio < 1 - UMBRAL_SEÑAL:
    senal = "EMITIR: el mercado cobra la volatilidad más cara de lo que GARCH espera (WM vende volatilidad cara)"
elif ratio > 1 + UMBRAL_SEÑAL:
    senal = "CAUTELA: GARCH ve más volatilidad que el mercado; subir la prima de conversión o esperar"
else:
    senal = "SIN VENTAJA clara: GARCH y el mercado coinciden; decide el ahorro de cupón, no la volatilidad"

ST = ST6                                    # trayectorias del mundo real (sección 8), 6 años
dil_sin = ACC_MM * np.maximum(ST - K_CONV, 0)          # USD mm entregados en acciones arriba del principal
dil_con = ACC_MM * np.maximum(ST - K_WARR, 0)
neto_cs = NETO_REPORTE
SENAL = dict(ratio=ratio, senal=senal, dil_sin=dil_sin.mean(), dil_con=dil_con.mean(), neto_cs=neto_cs,
             valor_fut_cs=neto_cs * np.exp(R_FREE * T_CONV))

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, np.percentile(dil_sin, 99), 60)
ax.hist(dil_sin[dil_sin > 0], bins=bins, color="firebrick", alpha=0.45, label="Sin call spread")
ax.hist(dil_con[dil_con > 0], bins=bins, color="darkgreen", alpha=0.45, label="Con call spread")
ax.set_title("WM: dilución al vencimiento en escenarios con conversión (GARCH, 6 años)")
ax.set_xlabel("USD mm entregados en acciones por encima del principal"); ax.legend()
mostrar("06_dilucion_call_spread")

print(f"σ GARCH 6 años {VOL_GARCH_6A:.1%}  |  σ implícita {IV_LSEG:.1%}  |  razón {ratio:.2f}")
print("Señal:", senal)
print(f"Dilución esperada al vencimiento: sin call spread USD {SENAL['dil_sin']:.1f} mm, con call spread USD {SENAL['dil_con']:.1f} mm")
print(f"Costo del call spread hoy USD {neto_cs:.1f} mm (≈ USD {SENAL['valor_fut_cs']:.1f} mm capitalizado a 6 años)")
print(f"P(se diluye sin call spread) = {(ST > K_CONV).mean():.1%} | P(se diluye con call spread) = {(ST > K_WARR).mean():.1%}")

σ GARCH 6 años 19.7%  |  σ implícita 22.7%  |  razón 0.87
Señal: EMITIR: el mercado cobra la volatilidad más cara de lo que GARCH espera (WM vende volatilidad cara)
Dilución esperada al vencimiento: sin call spread USD 307.5 mm, con call spread USD 197.8 mm
Costo del call spread hoy USD 54.5 mm (≈ USD 73.4 mm capitalizado a 6 años)
P(se diluye sin call spread) = 62.2% | P(se diluye con call spread) = 42.6%


## 13. Resumen e interpretación
- **GARCH vs árbol:** el árbol binomial valora la opción de conversión, el note hedge y el warrant; GARCH le dice
  **qué volatilidad usar** durante los 6 años de vida del convertible.
- **Pronóstico de la acción:** el Monte Carlo con GARCH da la probabilidad de que WM llegue al precio de conversión
  y al strike del warrant.
- **Riesgo:** el back testing verifica que el modelo no subestime pérdidas.
- **Para el reporte:** el cupón a la par con la volatilidad GARCH confirma (o ajusta) el 2.41 % del caso base.
- **Limitaciones:** GARCH(1,1) supone normalidad, no captura asimetría (GJR-GARCH, EGARCH) y a 6 años extrapola mucho
  más allá de los 5 años de datos. Las cifras con datos simulados no son de mercado.

In [19]:
print("RESUMEN")
print("Fuente de datos:", fuente)
for t in TICKERS:
    a, b = M[t]["p"][1], M[t]["p"][2]
    print(f"\n{t}: α={a:.3f}, β={b:.3f}, persistencia={a + b:.3f}; vol hoy {vol_anual_desde_var(M[t]['v_next']):.1%} "
          f"→ largo plazo {vol_anual_desde_var(M[t]['VL']):.1%}.")
    print(f"   Precio {S0[t]:.2f}; mediana a {HORIZON_DAYS} días {resumen_mc[t]['Mediana']:.2f} "
          f"(P5 {resumen_mc[t]['P5']:.2f} – P95 {resumen_mc[t]['P95']:.2f}).")
print(f"\nConvertible WM: σ GARCH 6 años {VOL_GARCH_6A:.1%} → cupón a la par (árbol T-F) "
      f"{cupon_tf['GARCH promedio 6 años']:.2%} vs. {cupon_tf['Implícita LSEG']:.2%} con la implícita de LSEG.")
print(f"   Call spread neto (convención del reporte): USD {SENAL['neto_cs']:.1f} mm. Señal: {SENAL['senal']}")
print(f"   P(WM > ${K_WARR:.0f} en 6 años) = {(ST6 > K_WARR).mean():.1%}")

RESUMEN
Fuente de datos: Yahoo Finance (yfinance)

WM: α=0.135, β=0.543, persistencia=0.678; vol hoy 24.7% → largo plazo 19.7%.
   Precio 237.71; mediana a 126 días 245.14 (P5 193.75 – P95 307.26).

RSG: α=0.265, β=0.450, persistencia=0.715; vol hoy 24.6% → largo plazo 19.6%.
   Precio 245.97; mediana a 126 días 254.15 (P5 201.98 – P95 319.79).

Convertible WM: σ GARCH 6 años 19.7% → cupón a la par (árbol T-F) 2.85% vs. 2.41% con la implícita de LSEG.
   Call spread neto (convención del reporte): USD 54.5 mm. Señal: EMITIR: el mercado cobra la volatilidad más cara de lo que GARCH espera (WM vende volatilidad cara)
   P(WM > $367 en 6 años) = 42.6%


In [20]:
# (Opcional) Descargar los precios de WM para el repositorio y tomar captura de las tablas para capturas/17_Colab_GARCH.png
try:
    from google.colab import files
    if os.path.exists("datos/WM_precios.csv"):
        files.download("datos/WM_precios.csv")
except Exception:
    pass